# CXAS SCRAPI quick tryout
This notebook installs `cxas-scrapi` and runs a minimal sanity check.

Docs: https://googlecloudplatform.github.io/cxas-scrapi/latest/

In [1]:
# Install (safe to re-run)
%pip -q install cxas-scrapi

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Quick sanity check: version + import name
import importlib
import importlib.util
import sys

def find_dist_version(dist_name: str) -> str | None:
    try:
        from importlib.metadata import version
        return version(dist_name)
    except Exception:
        return None

print('python:', sys.version.split()[0])
print('cxas-scrapi:', find_dist_version('cxas-scrapi'))

# Try to locate an importable top-level module (best-effort; package may expose different names)
candidates = ['cxas', 'cxas_scrapi', 'scrapi']
for name in candidates:
    if importlib.util.find_spec(name) is not None:
        m = importlib.import_module(name)
        print('import ok:', name, '->', getattr(m, '__version__', '(no __version__)'))
        break
else:
    print('No known top-level module found from', candidates)

python: 3.14.0
cxas-scrapi: 1.3.0
import ok: cxas_scrapi -> (no __version__)


## Create an App + Agent (SCRAPI)
This follows the SCRAPI guide “Creating Agents”. You must set `PROJECT_ID` and have Google Cloud credentials available (ADC).

In [ ]:
# ---- Configure these ----
PROJECT_ID = "YOUR_GCP_PROJECT_ID"  # e.g. "my-gcp-project"
LOCATION = "us"  # CXAS location/region, e.g. "us" or "us-central1"

# Your desired model
MODEL = "gemini-3.1-flash-live"

APP_ID = "my-support-agent"  # lowercase, hyphens, numbers
APP_DISPLAY_NAME = "My Support Agent"
AGENT_ID = "support-root"
AGENT_DISPLAY_NAME = "Support Root Agent"

# Safety: refuse to run if you forgot to set the project id
if PROJECT_ID == "YOUR_GCP_PROJECT_ID":
    raise ValueError("Set PROJECT_ID before running this cell.")

In [ ]:
# Preflight: if model isn't enabled in this region, fail fast with a clear hint
try:
    assert isinstance(MODEL, str) and MODEL.strip(), "MODEL must be a non-empty string"
except Exception as e:
    raise
print(f"Using LOCATION={LOCATION!r}, MODEL={MODEL!r}")
print("If you see a 'model ... is not available in <region>' error later,")
print("it means CXAS hasn't enabled that model in that region for your project yet.")

In [ ]:
from cxas_scrapi.core.apps import Apps
from cxas_scrapi.core.agents import Agents

apps = Apps(project_id=PROJECT_ID, location=LOCATION)

app = apps.create_app(
    app_id=APP_ID,
    display_name=APP_DISPLAY_NAME,
    description="Created from a notebook using cxas-scrapi",
)

print("app:", app.name)

app_name = app.name  # full resource name

In [ ]:
agents = Agents(app_name=app_name)

create_kwargs = {}
if MODEL:
    create_kwargs["model"] = MODEL

agent = agents.create_agent(
    agent_id=AGENT_ID,
    display_name=AGENT_DISPLAY_NAME,
    **create_kwargs,
 )

print("agent:", agent.name)

# Set as the root agent for the app
apps.update_app(
    app_name=app.name,
    root_agent=agent.name,
 )

print("root agent set")

In [ ]:
# Optional: create a simple Python Tool and attach it to the agent
from cxas_scrapi.core.tools import Tools

tools = Tools(app_name=app_name)

tool = tools.create_tool(
    tool_id="hello_tool",
    display_name="hello_tool",
    description="Returns a friendly greeting.",
    payload={
        "python_code": """\
from datetime import datetime

def hello_tool(name: str | None = None) -> dict:
    """Return a friendly greeting."""
    who = name or "there"
    return {
        "message": f"Hello, {who}!",
        "timestamp": datetime.utcnow().isoformat() + "Z",
    }
""",
    },
)

# Attach tool + include end_session built-in tool on root agent
end_session_tool = f"{app_name}/tools/end_session"
agents.update_agent(
    agent_name=agent.name,
    tools=[tool.name, end_session_tool],
 )

print("tool created:", tool.name)
print("agent updated with tools")

## Feasibility: create → deploy → evaluate (SCRAPI)
Below is a minimal lifecycle checklist you can run with SCRAPI.

### 1) Create (platform resources)
- Python: `Apps.create_app`, `Agents.create_agent`, `Tools.create_tool` (see cells above)
- CLI: `cxas create`

> Note: “deploy” in CXAS typically means “push the latest agent config/code to the platform”, not deploying infra.

### 2) Deploy / Promote changes (pull/push workflow)
- `cxas pull <app>` downloads the app into `cxas_app/<AppName>/...`
- Edit `instruction.txt`, agent JSON, tool/callback `python_code.py` locally
- `cxas lint` validates best practices (60+ rules)
- `cxas push cxas_app/<AppName>` pushes changes back
- Promotion: `cxas push ... --to projects/{project}/locations/{location}/apps/{target}` (target must already exist)

> SCRAPI treats the platform as the source of truth; pushing overwrites remote state for changed resources.

### 3) Evaluate
SCRAPI supports 5 evaluation types:
- Platform Goldens (platform) — YAML; upload with `cxas push-eval`, run with `cxas run --wait`
- Local Simulations (local via Sessions API) — YAML; run via CLI / Python classes
- Tool Tests (local) — YAML; run with `cxas test-tools`
- Callback Tests (local) — pytest; run with `cxas test-callbacks`
- Turn Evals (local via Sessions API) — Python-driven assertions

> For CI-style end-to-end validation, use `cxas ci-test` (temporary app) or `cxas local-test` (Docker).